[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/magnetic_resonance/magnetic_resonance.ipynb)

# Magnetic Resonance

Spins in a magnetic field precess about it: electron spins in electron spin resonance, nuclei in NMR and MRI, and the spins used as qubits. A radio-frequency field near the precession frequency tips them. Left to themselves they relax, back toward the field over a time T1, and out of step with each other over a time T2. This notebook computes three things an experimenter measures: how a driven spin moves, the absorption line and how strong driving broadens it, and the spin echo that separates true dephasing from an uneven field. The last section builds a whole echo experiment as one map from spin state to spin state.

Times are in microseconds and frequencies in radians per microsecond, with T1 = 10 µs and T2 = 4 µs, typical of an electron spin in a solid.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Generator, Iterator

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.quantum.magnetic_resonance import render

np.set_printoptions(precision=3, suppress=True)

# The geometric algebra of three-dimensional space.
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
# A spin's state is its own reverse: a scalar plus a vector, (1 + Bloch vector) / 2.
State = ga.gatype.self_reverse()                       # 1 x y z
Rates = ga.gatype((State, State))                      # State <- State
# What becomes of each state over a stretch of time.
Evolution = ga.gatype((State, State))                  # State <- State
# A relaxation process is a vector plus a bivector, v + I w.
Process = ga.gatype(ga.subspace.vector() + ga.subspace.bivector())   # x y z xy xz yz
Rotor = ga.gatype.rotor()
# The pseudoscalar squares to minus one and commutes with everything.
I = mv.xyz                                             # [] Pseudoscalar
one = mv.scalar([1.0])                                 # [] Scalar
T1, T2 = 10.0, 4.0                                     # µs

## 1. The state and how it changes

The state of a spin, or of many identical spins, is half of one plus a vector r, the Bloch vector. For a single spin r has length one and points along the spin; for a mixture it is shorter. In the frame that turns with the drive, the fields act through a vector H, half the detuning along z plus half the drive strength along x, and they turn the state by −I times its commutator with H. Relaxation is written with multivectors L, each adding L ρ L̃ − (L̃L ρ + ρ L̃L)/2, with L̃ the reverse of L. One of them is the vector x times the state of a spin against the field: it takes that part of the state and turns it over onto the field, at the rate 1/T1. One along z scrambles the phase. Leaving the state open turns all of this into one linear map, the generator.

In [ ]:
def state(bloch: Vector) -> State:
    """The state with the given Bloch vector."""
    return 0.5 * (one + bloch)                                                  # [...] State


def relaxation(process: Process) -> Rates:
    """The change of a state due to a relaxation process L, as a map: the state is left open."""
    back = process.reverse().symmetric_reverse_product()                        # [] State
    return process * State * process.reverse() - 0.5 * (back * State + State * back)   # State <- State


# x times the state of a spin against the field: it turns that part of the state over onto the field.
raise_toward_field = mv.x * state(-mv.z)                                        # [] Process


def generator(detuning: Scalar, drive: Scalar) -> Rates:
    """The rate of change of a state, in the frame that turns with the drive."""
    hamiltonian = 0.5 * (mv.z * detuning + mv.x * drive)                        # [...] Vector
    turning = -I * (hamiltonian * State - State * hamiltonian)                  # [...] State <- State
    relaxing = (1 / T1) * relaxation(raise_toward_field) + 0.5 * (1 / T2 - 1 / (2 * T1)) * relaxation(mv.z)   # State <- State
    return (turning + relaxing).cast(State)                                     # [...] State <- State


# A detuning of 0.7 and a drive of 1.3 rad/µs, acting on the state with Bloch vector r.
rates = generator(mv.scalar([0.7]), mv.scalar([1.3]))                           # [] State <- State
r = np.array([0.4, -0.2, 0.6])                                                  # [3]
change = rates(state(mv.vector(r)))                                             # [] State

In [ ]:
# The change of the Bloch vector, against the Bloch equations dr/dt = w x r - relaxation, w = (drive, 0, detuning):
print("generator:      ", render.components(change))
print("Bloch equations:", np.cross([1.3, 0.0, 0.7], r) - np.array([r[0] / T2, r[1] / T2, (r[2] - 1) / T1]))

For comparison, in matrix notation the state reads as a 2×2 Hermitian density matrix, the reverse as the Hermitian conjugate, I as the imaginary unit i, x times the state against the field as the raising operator σ₊, and the generator as the Lindblad superoperator, a 4×4 matrix acting on the flattened density matrix. Restricted to the Bloch vector it reads as the Bloch equations.

## 2. The steady state

Under a steady drive a spin turns about the drive's axis, (drive, 0, detuning), while relaxation pulls it back, and it spirals into a state that no longer changes. That state is where the generator vanishes. The generator never changes the scalar part of a state, so it is singular. Adding the scalar part as a dyad on one pins it, and a single solve gives the steady state. Over a short time dt a state evolves by exp(dt L), with L the generator; to fourth order in dt that is a sum of powers of the generator, a map built once and applied at every step. The picture follows three spins at different detunings from the field axis into the states the solve predicts.

In [ ]:
def steady(rates: Rates) -> State:
    """The state that no longer changes, with its scalar part pinned to one half."""
    scalar_part = one.scalar_product(State)                                    # Scalar <- State
    return (rates + one * scalar_part).solve(one * 0.5)                         # [...] State


def evolution(rates: Rates, dt: float) -> Evolution:
    """What becomes of each state over a time dt: exp(dt L), to fourth order in dt."""
    small = rates * dt                                                          # [...] State <- State
    term = total = State
    for order in range(1, 5):
        # The next term of the series, (dt L)^order / order!, by composition.
        term = small(term) / order                                              # [...] State <- State
        total = total + term
    return total                                                                # [...] State <- State


def evolve(each_step: Evolution, rho: State, steps: int) -> Generator[State, None, State]:
    """The state before each of the given number of steps; returns the state after the last."""
    for _ in range(steps):
        yield rho                                                               # [...] State
        rho = each_step(rho)
    return rho


detunings = np.array([0.0, 0.5, 1.0])                                           # rad/µs
driven = generator(mv.scalar(detunings[:, None]), mv.scalar([1.0]))            # [detunings] State <- State
# All spins start along the field and are driven for 40 µs.
along_field = state(mv.z).broadcast_to(detunings.shape)                         # [detunings] State
settled = steady(driven)                                                        # [detunings] State
render.draw_nutation(evolve(evolution(driven, 0.02), along_field, 2000), settled);

## 3. Line shapes

A magnetic resonance spectrum records how much energy the spins absorb from the drive at each detuning. That is the part of the steady Bloch vector a quarter turn behind the drive, −r_y; the part in step with it, r_x, is the dispersion. Because the steady state is a single solve, the whole spectrum for several drive strengths is one batched call. With a weak drive the absorption line has a half-width of 1/T2. Driven harder, the spins cannot keep up: the line saturates and broadens, and its half-width grows to √(1 + Ω²T1T2)/T2.

In [ ]:
sweep = np.linspace(-3.0, 3.0, 241)                                              # detunings, rad/µs
drives = np.array([0.05, 0.3, 1.0])                                              # rad/µs
spectrum = steady(generator(mv.scalar(sweep[:, None, None]), mv.scalar(drives[None, :, None])))   # [sweep, drives] State
render.draw_lines(sweep, drives, spectrum);

For comparison, the textbook solution of the Bloch equations reads r_z = (1 + Δ²T2²)/d and −r_y = Ω T2/d, with d = 1 + Δ²T2² + Ω²T1T2; the checks at the end compare against it.

## 4. Spin echo

In a real sample the field is not quite uniform, so each spin has its own detuning. Tipped into the transverse plane by a short pulse, the spins fan out, and their mean, the signal a pick-up coil sees, vanishes long before T2. A second pulse that turns every spin half a turn about x reverses the fan: each spin retraces its drift, and at twice the delay they line up again. By then only the true dephasing is lost, a factor exp(−2τ/T2). A short pulse is a rotor, and it acts on a state by its sandwich.

In [ ]:
def pulse(angle: float) -> Rotor:
    """A short strong pulse along x, turning the Bloch vector about x by the angle."""
    return (I * mv.x * (-angle / 2)).exp()                                      # [] Rotor


def echo(each_step: Evolution, rho: State, before: int, after: int) -> Iterator[State]:
    """The states through a spin echo: tipped onto -y, left for the given number of steps, turned
    half a turn about x, and left again."""
    rho = yield from evolve(each_step, pulse(np.pi / 2) >> rho, before)
    yield from evolve(each_step, pulse(np.pi) >> rho, after)


spread, delay, dt = 2.0, 2.5, 0.01                                               # rad/µs, µs, µs
scattered = np.random.default_rng(0).normal(0.0, spread, 300)                    # [spins] rad/µs
# No drive between the pulses; the spins start along the field and are followed for 8 µs in all.
free = generator(mv.scalar(scattered[:, None]), mv.scalar([0.0]))                # [spins] State <- State
along_field = state(mv.z).broadcast_to(scattered.shape)                         # [spins] State
waits = round(delay / dt)
# The spins seen from above the field, one frame every 8 steps, beside the signal so far.
frames = render.animate_echo(scattered, echo(evolution(free, dt), along_field, waits, 800 - waits), dt, delay, T2, 8)
display(Image(filename=save_animation(frames, "resonance_echo", 40)))

## 5. The experiment as one map

The evolution over one step is a map from state to state. Composed with itself it spans two steps, then four, and so on. The pulses are maps too, their sandwiches with the state left open, so a pulse sequence is a composition of maps, built before any state is put in. Each spin has its own map, and the sample's is their average: one map from spin state to spin state for the whole experiment. Built for delays from 10 ns to 10 µs at once, the free decay is gone within a microsecond, while the echo fades only at the rate 1/T2.

In [ ]:
def doublings(span: Evolution, count: int) -> Iterator[Evolution]:
    """The evolution over its own span, then over twice and four times that span, and so on."""
    for _ in range(count):
        yield span
        span = span(span)


# The evolution over one step, doubled for delays from dt to 1024 dt.
waiting = stack(list(doublings(evolution(free, dt), 11)))                       # [delays, spins] State <- State
# The times after the first pulse.
times = 2 * dt * 2.0 ** np.arange(11)                                            # µs
# The pulses as maps: their sandwiches with the state left open.
tip = pulse(np.pi / 2) >> State                                                  # [] State <- State
turn = pulse(np.pi) >> State                                                     # [] State <- State
# Tip, wait, turn, wait: the echo. Tip and wait twice as long: the free decay.
echo = waiting(turn(waiting(tip)))                                               # [delays, spins] State <- State
decay = waiting(waiting(tip))                                                    # [delays, spins] State <- State
# Every spin has its own map; the sample's is their average.
sample_echo = echo.mean(axis=-1)                                                 # [delays] State <- State
sample_decay = decay.mean(axis=-1)                                               # [delays] State <- State
echoed = sample_echo(state(mv.z))                                                # [delays] State
faded = sample_decay(state(mv.z))                                                # [delays] State
render.draw_decays(times, echoed, faded, T2);

In [ ]:
# checks
# The generator is the Bloch equations; the steady states do not change and are the textbook ones;
# the echo built as a map has lost only the true dephasing.
bloch_equations = np.cross([1.3, 0.0, 0.7], r) - np.array([r[0] / T2, r[1] / T2, (r[2] - 1) / T1])
np.testing.assert_allclose(render.components(change), bloch_equations, atol=1e-12)
np.testing.assert_allclose(driven(settled).kernel, 0.0, atol=1e-12)
D, W = sweep[:, None], drives[None, :]
d = 1 + (D * T2) ** 2 + W**2 * T1 * T2
components = render.components(spectrum)
np.testing.assert_allclose(components[..., 2], (1 + (D * T2) ** 2) / d, atol=1e-12)
np.testing.assert_allclose(-components[..., 1], W * T2 / d, atol=1e-12)
np.testing.assert_allclose(render.transverse(echoed), np.exp(-times / T2), rtol=1e-5)